In [ ]:
import os
import sys
if 'google.colab' in sys.modules:
  !pip install -q numba imageio
  !apt-get install -y -qq ffmpeg

os.makedirs('../results/gifs', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from io import BytesIO

In [ ]:
class czastka:
  def __init__(self, pos, vel):
    self.r = pos.astype(float)
    self.v = vel.astype(float)
    self.a = np.zeros(2, dtype=float)

In [ ]:
def draw_with_ghosts(ax, pos_all, box_size, r=0.3, s=1160, color='royalblue'):
  ax.scatter(pos_all[:, 0], pos_all[:, 1], s=s, color=color)

  left  = pos_all[pos_all[:, 0] < r] + [box_size, 0]
  right = pos_all[pos_all[:, 0] > box_size - r] - [box_size, 0]
  bottom = pos_all[pos_all[:, 1] < r] + [0, box_size]
  top    = pos_all[pos_all[:, 1] > box_size - r] - [0, box_size]

  ghosts = [left, right, bottom, top]
  ghosts = [g for g in ghosts if len(g)]
  if ghosts:
    g = np.vstack(ghosts)
    ax.scatter(g[:, 0], g[:, 1], s=s, color=color)


def calculate_forces(flat_particles, box_size, eps, sigma, rC):
  N = len(flat_particles)
  virial_sum = 0
  potential_energy = 0

  for p in flat_particles:
    p.a = np.zeros(2)

  for a_idx in range(N):
    i = flat_particles[a_idx]

    for b_idx in range(a_idx + 1, N):
      j = flat_particles[b_idx]

      r = i.r - j.r
      r -= box_size * np.round(r / box_size)
      dist = np.linalg.norm(r)

      if dist < rC:
        F = ((48 * eps) / dist**2) * ((sigma / dist)**12 - 0.5 * (sigma / dist)**6) * r
        potential_energy += 4 * eps * ((sigma / dist)**12 - (sigma / dist)**6)

        i.a += F
        j.a -= F
        virial_sum += np.dot(r, F)

  return virial_sum, potential_energy

In [ ]:
n = 4
N_particles = n**2
box_size = 8.0
eps, sigma, kB, mass = 1, 1, 1, 1
radius = sigma / 2
dt = 1e-4
T_ext = 2.5
steps = 5*10**4 + 1

rC = 2.5 * sigma

In [ ]:
sigma_v = np.sqrt(kB * T_ext / mass)
vel = np.random.normal(0, sigma_v, size=(n, n, 2))
vel -= np.mean(vel, axis=(0, 1))
T_actual = mass * np.mean(vel**2) / (2 * kB)
vel *= np.sqrt(T_ext / T_actual)

pos = (np.array([[[i, j] for j in range(n)] for i in range(n)]) / n + np.array([1, 1]) / (2 * n)) * box_size

particles = [[czastka(pos[i, j], vel[i, j]) for j in range(n)] for i in range(n)]
flat = np.ravel(particles)

virial = calculate_forces(flat, box_size, eps, sigma, rC)

# v(t_0 - dt/2) = v(t_0) - F(t_0)/m * dt/2
for p in flat:
  p.v -= (p.a / mass) * dt / 2

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

Ep_history = np.zeros(steps)
Temp_history = np.zeros(steps)
Press_history = np.zeros(steps)
Time_history = np.arange(steps) * dt

with imageio.get_writer('../results/gifs/06_thermostat.gif', mode='I', duration=0.05) as writer:
  for k in range(steps): # 20000 -> ~30s
    Kinetic_u = 0
    v_u_list = []

    for p in flat:
      v_u = p.v + (p.a / mass) * (dt / 2)
      v_u_list.append(v_u)
      Kinetic_u += 0.5 * mass * np.sum(v_u**2)

    Temp_inst = Kinetic_u / (kB * N_particles)
    eta = np.sqrt(T_ext / Temp_inst)

    v_old = np.array([p.v for p in flat])
    forces = np.array([p.a for p in flat])
    v_new = (2 * eta - 1) * v_old + eta * (forces / mass) * dt

    for idx, p in enumerate(flat):     
      p.v = v_new[idx]
      p.r += p.v * dt
      p.r %= box_size # PBC
      
    v_t = (v_old + v_new) / 2
    Ek_final = 0.5 * mass * np.sum(v_t**2)
    Temp_history[k] = Ek_final / (kB * N_particles)

    virial, Ep = calculate_forces(flat, box_size, eps, sigma, rC)
    Ep_history[k] = Ep / N_particles

    V = box_size**2
    Press_history[k] = (N_particles * kB * Temp_history[k]) / V + (virial / (2 * V))

    if k % 500 == 0:
      ax.clear()
      pos_all = np.array([p.r.copy() for p in flat])
      draw_with_ghosts(ax, pos_all, box_size, r=radius, s=1160)
      ax.set_xlim(0, box_size)
      ax.set_ylim(0, box_size)
      ax.set_aspect('equal')
      ax.set_title(f'Step: {k}')
      
      buf = BytesIO()
      plt.savefig(buf, format='png')
      buf.seek(0)
      writer.append_data(imageio.imread(buf))

plt.close(fig)

In [ ]:
skip = 100

fig_plots, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

ax1.plot(Time_history[::skip], Temp_history[::skip], color='red', label='Temperature')
ax1.axhline(T_ext, color='black', linestyle='--', label='Target T (Thermostat)')
ax1.set_ylabel('Temperature')
ax1.set_title('Izokinetic Thermostat')
ax1.legend()

ax2.plot(Time_history[::skip], Press_history[::skip], color='blue', label='Pressure')
ax2.set_ylabel('Pressure')
ax2.set_xlabel('Time')
ax2.set_title('Pressure Evolution')
ax2.legend()

ax3.plot(Time_history[::skip], Ep_history[::skip] + Temp_history[::skip] * kB, label='Total Energy / N')
ax3.set_ylabel('Total Energy / N')
ax3.set_xlabel('Time')
ax3.set_title('Energy over time')
ax3.legend()

plt.tight_layout()
plt.savefig('../results/plots/06_thermostat_energy_stability.png')
plt.show()